# Confirmation of Jev's responsiveness result: draw the sample to label

Runs on a free Colab **CPU** runtime in about 5 minutes; no GPU and no token are needed.
It follows the locked protocol in `docs/annotation/confirmation_protocol.md` and ends by
downloading `confirmation_sample.zip`.

1. **Raw data.** `scripts/fetch_booking_515k.py` downloads the Booking 515K CSV and
   refuses it unless its size and SHA-256 match the Kaggle file.
2. **Sample.** `scripts/confirmation_sample.py draw` draws 400 new texts from the negative
   fields of the 863 compared hotels: 200 random texts from February-July 2016 and 200 from
   February-July 2017. The pilot's 300 texts are left out.

**After the download:**
- **`sheet.csv` is for you: 400 texts.** Fill in only the `responsiveness` column. Write `1`,
  `0` or `unsure`, as the responsiveness section of `complaint_topics_guideline.md` says, and
  write `yes` in `done`. A blank cell sends the whole sheet back.
- **Do not send these texts to any service** (Jev, a chat assistant, anything) before you have
  handed the sheet in.
- **Do not open `key.csv`.** It says which texts the lexicon matched.
- **Send back** the filled sheet, `key.csv` and `manifest.json`, as they are.

In [ ]:
# The commit that locked the protocol. The sample is drawn from exactly this code.
REVISION = "6a47107be66f3e17273646eb9bdcf260f8b9d99d"

!git clone --quiet https://github.com/krimits/hotel-review-nlp.git
%cd hotel-review-nlp
!git checkout --quiet {REVISION}
!git log --oneline -1
!pip install --quiet --no-deps -e .

In [ ]:
import subprocess


def run(command: str) -> None:
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


run("python scripts/fetch_booking_515k.py")

## Draw the sample (seed 1, as the protocol fixes)

In [ ]:
run("python scripts/confirmation_sample.py draw --expect-hotels 863 --out runs/confirmation")

In [ ]:
import json

manifest = json.load(open("runs/confirmation/manifest.json"))
pilot = json.load(open("docs/case_study/results/annotation/sample_manifest.json"))
print("protocol", manifest["protocol_sha256"][:12], "| code", manifest["code_commit"][:7],
      "| hotels", manifest["hotels_compared"], "| texts", manifest["items"],
      "| pilot texts left out", manifest["design"]["left_out"]["texts"])
for name, numbers in manifest["periods"].items():
    drawn_in_pilot = pilot["periods"][name]["R"] + pilot["periods"][name]["M_texts"]
    print(name, "texts", numbers["population"], "(the pilot's:", pilot["periods"][name]["population"], ")",
          "| pilot texts among them", numbers["excluded_pilot_texts"], "of", drawn_in_pilot,
          "| drawn", numbers["drawn"], "| set aside by the filter", numbers["excluded_by_filter"])

## Download the sheet, the key and the guideline

In [ ]:
import zipfile

from google.colab import files

paths = ["runs/confirmation/sheet.csv", "runs/confirmation/key.csv", "runs/confirmation/manifest.json",
         "docs/annotation/complaint_topics_guideline.md"]
with zipfile.ZipFile("confirmation_sample.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        archive.write(path, arcname=path.rsplit("/", 1)[-1])
files.download("confirmation_sample.zip")